**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 42: 프로젝트 배포 (FastAPI + Streamlit)

Session 40에서 학습한 에이전트 어댑터를 **실제 서비스로 배포**합니다.

### 흐름
```
1. 에이전트 어댑터 로드 및 확인
2. FastAPI 서버 코드 생성 → 터미널에서 실행
3. API 테스트
4. Streamlit 챗봇 생성 → 터미널에서 실행
```

In [ ]:
# =====================================================
# 📦 프로젝트 설정 및 Agent 산출물 확인
# =====================================================
import json, os, sys

OUTPUT_DIR = "./outputs/agent"   # Session 40 산출물 디렉터리

config_path = os.path.join(OUTPUT_DIR, "agent_config.json")
kb_path     = os.path.join(OUTPUT_DIR, "agent_kb.pkl")
pipeline_py = "agent_pipeline.py"

# project_plan.json (39 산출물 — 옵션)
plan_path = "project_plan.json"
if os.path.exists(plan_path):
    project_plan = json.load(open(plan_path, encoding="utf-8"))
else:
    project_plan = {
        "project_name": "Agent Project",
        "task": "MCP + LangGraph + A2A 통합 에이전트",
        "data_config": {"system_prompt": "당신은 AI/ML 전문가입니다."},
        "eval_prompts": ["MCP 가 뭐야?", "A2A 의 특징은?"],
    }

print(f"📂 Session 40 산출물 확인 ({OUTPUT_DIR}/)")
for name, path in [("agent_config.json", config_path),
                    ("agent_kb.pkl",      kb_path),
                    ("agent_pipeline.py", pipeline_py)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'} {name}: {path}")

if os.path.exists(config_path):
    cfg = json.load(open(config_path, encoding="utf-8"))
    print(f"\n📋 Agent 메타데이터:")
    for k, v in cfg.items():
        print(f"  {k:18s} = {v}")
else:
    print("\n❌ agent_config.json 없음 — Session 40 을 먼저 실행하세요")


---
## 1️⃣ FastAPI 서버 생성

학습된 모델을 API로 서빙합니다.


In [ ]:
# =====================================================
# 📝 FastAPI 서버 코드 생성 — AgentPipeline 서빙
# =====================================================

server_code = f'''
# project_server.py - Agent API 서버 (Session 40 AgentPipeline 서빙)
# 실행: python project_server.py

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional
import time
import sys, os
import uvicorn

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from agent_pipeline import AgentPipeline

app = FastAPI(
    title="{project_plan['project_name']}",
    description="{project_plan['task']}",
    version="1.0.0",
)


class ChatRequest(BaseModel):
    messages: List[dict]
    max_new_tokens: Optional[int] = 256
    temperature: Optional[float] = 0.7


class ChatResponse(BaseModel):
    response: str
    route: str
    time_seconds: float


agent = AgentPipeline()


@app.get("/health")
async def health():
    return {{"status": "ok", "n_docs": len(agent.docs)}}


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    # 마지막 user message 추출
    user_msg = next((m["content"] for m in reversed(req.messages)
                     if m["role"] == "user"), "")
    t0 = time.time()
    result = agent.run(user_msg)
    return ChatResponse(
        response=result["answer"],
        route=result["route"],
        time_seconds=round(time.time() - t0, 3),
    )


# Agent Card (A2A 호환) — 다른 에이전트가 발견 가능
@app.get("/.well-known/agent.json")
async def agent_card():
    return {{
        "name": "{project_plan['project_name']}",
        "description": "{project_plan['task']}",
        "url": "http://localhost:9200/chat",
        "capabilities": {{"streaming": False}},
        "skills": [{{"id": "qa", "name": "지식·계산·시각",
                     "inputModes": ["text"]}}],
    }}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9200)
'''

server_path = "project_server.py"
with open(server_path, "w", encoding="utf-8") as f:
    f.write(server_code)

print(f"✅ {server_path} 생성 완료")
print(f"\n실행: python {server_path}")
print("이후 cell 6 (API 테스트) 실행 가능")


---
## 2️⃣ API 테스트

위 서버를 터미널에서 실행한 후, 아래 셀로 테스트합니다.


In [ ]:
# =====================================================
# 🧪 API 테스트
# ⚠️ 터미널에서 project_server.py가 실행 중이어야 합니다
# =====================================================
import requests

API_URL = "http://localhost:9200"

# 서버 상태 확인
try:
    health = requests.get(f"{API_URL}/health", timeout=5).json()
    print(f"✅ 서버 상태: {health}")
except:
    print("❌ 서버가 실행 중이 아닙니다.")
    print(f"   터미널에서: python {os.path.join(OUTPUT_DIR, 'project_server.py')}")

# 질문 테스트
test_prompts = project_plan["eval_prompts"][:3]

for prompt in test_prompts:
    try:
        response = requests.post(
            f"{API_URL}/chat",
            json={
                "messages": [
                    {"role": "system", "content": project_plan["data_config"]["system_prompt"]},
                    {"role": "user", "content": prompt}
                ],
                "max_new_tokens": 200,
            },
            timeout=60,
        )
        result = response.json()
        print(f"\n🔹 질문: {prompt}")
        print(f"   응답: {result['response'][:150]}")
        print(f"   시간: {result['time_seconds']}초, 토큰: {result['tokens_generated']}")
    except Exception as e:
        print(f"\n❌ 에러: {e}")


---
## 3️⃣ Streamlit 챗봇 생성

API 서버와 연동하는 웹 채팅 인터페이스를 만듭니다.


In [ ]:
# =====================================================
# 📝 Streamlit 챗봇 코드 생성
# =====================================================

streamlit_code = f"""
# project_chatbot.py - 파인튜닝 모델 챗봇
# 실행: streamlit run project_chatbot.py --server.port 9300

import streamlit as st
import requests

API_URL = "http://localhost:9200"
SYSTEM_PROMPT = "{project_plan['data_config']['system_prompt']}"

st.set_page_config(page_title="{project_plan['project_name']}", page_icon="🤖")
st.title("🤖 {project_plan['project_name']}")
st.caption("{project_plan['task']}")

# 채팅 이력
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 메시지 표시
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# 사용자 입력
if prompt := st.chat_input("질문을 입력하세요..."):
    st.session_state.messages.append({{"role": "user", "content": prompt}})
    with st.chat_message("user"):
        st.write(prompt)
    
    # API 요청
    with st.chat_message("assistant"):
        with st.spinner("생각 중..."):
            try:
                messages = [
                    {{"role": "system", "content": SYSTEM_PROMPT}},
                ] + st.session_state.messages
                
                response = requests.post(
                    f"{{API_URL}}/chat",
                    json={{"messages": messages, "max_new_tokens": 300}},
                    timeout=60,
                )
                result = response.json()
                answer = result["response"]
            except Exception as e:
                answer = f"❌ 서버 연결 실패: {{e}}"
        
        st.write(answer)
        st.session_state.messages.append({{"role": "assistant", "content": answer}})

# 사이드바
with st.sidebar:
    st.markdown("### ℹ️ 정보")
    st.markdown(f"- 모델: `{MODEL_NAME}`")
    st.markdown(f"- 도메인: {project_plan['domain']}")
    if st.button("대화 초기화"):
        st.session_state.messages = []
        st.rerun()
"""

chatbot_path = os.path.join(OUTPUT_DIR, "project_chatbot.py")
with open(chatbot_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print(f"✅ 챗봇 코드 생성: {{chatbot_path}}")
server_path = os.path.join(OUTPUT_DIR, 'project_server.py')
print(f"\n🚀 실행 방법:")
print(f"   1. 터미널 1: python {server_path}")
print(f"   2. 터미널 2: streamlit run {chatbot_path} --server.port 9300")
print(f"   3. 브라우저: http://localhost:9300")


In [ ]:
# =====================================================
# 🚀 Streamlit 챗봇 실행
# =====================================================
import subprocess

chatbot_path = os.path.join(OUTPUT_DIR, "project_chatbot.py")
cmd = f"streamlit run {chatbot_path} --server.port 9300"

print("🚀 아래 명령어를 터미널에서 실행하세요:")
print("="*60)
print(f"  {cmd}")
print("="*60)
print(f"\n브라우저에서 http://localhost:9300 접속")
print(f"⚠️ project_server.py(포트 9200)가 먼저 실행 중이어야 합니다")


---
## 📝 프로젝트 최종 정리


In [ ]:
# =====================================================
# 🎉 프로젝트 최종 요약
# =====================================================
print("🎉 Agent 프로젝트 완료!")
print("="*60)

files = {
    "Agent 설정":       os.path.join(OUTPUT_DIR, "agent_config.json"),
    "Agent KB":         os.path.join(OUTPUT_DIR, "agent_kb.pkl"),
    "Agent 모듈":       "agent_pipeline.py",
    "MCP 서버":         "mcp_server.py",
    "A2A 서버":         "a2a_server.py",
    "API 서버":         "project_server.py",
    "챗봇 앱":          "project_chatbot.py",
}

for name, path in files.items():
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {name}: {path}")

print(f"\n📋 실행 방법:")
print(f"   터미널 1: python project_server.py")
print(f"   터미널 2: streamlit run project_chatbot.py --server.port 9300")
print(f"   브라우저: http://localhost:9300")
print(f"\n💡 Agent Card 확인: curl http://localhost:9200/.well-known/agent.json")
